In [0]:
%pip install xgboost -q

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import mlflow
import numpy as np
import pandas as pd

# Load prediction data from catalog
pred_df = spark.table("science_home.`ml-dielectric`.prediction_data").toPandas()

# Separate identifiers and features
ids = pred_df['id']
formulas = pred_df['formula']

# Prepare features (drop non-feature columns)
feature_cols = [c for c in pred_df.columns if c not in ['id', 'formula', 'dielectric_constant']]
X_pred = pred_df[feature_cols]

# Load the latest version of the registered model from Unity Catalog
model_name = "science_home.ml-dielectric.xgboost_dielectric"
client = mlflow.MlflowClient()
latest_version = max(v.version for v in client.search_model_versions(f"name='{model_name}'"))
model_uri = f"models:/{model_name}/{latest_version}"
print(f"Loading model version: {latest_version}")
model = mlflow.pyfunc.load_model(model_uri)

# Predict (model outputs log-transformed values)
y_pred_log = model.predict(X_pred)

# Inverse log transform to get actual dielectric constant
y_pred = np.expm1(y_pred_log)  # inverse of log1p

# Create final output DataFrame
results = pd.DataFrame({
    'id': ids,
    'formula': formulas,
    'predicted_dielectric_constant': y_pred
})

display(results)

Loading model version: 4


id,formula,predicted_dielectric_constant
mp-22546,CoAsRh,28.492823
mp-22548,Th2CoB10,22.316124
mp-22549,SmCoO3,20.69673
mp-22550,BaDy2CuO5,19.049324
mp-22551,UP2(PbO5)2,18.420465
mp-22552,Ni(AuF4)2,5.7001324
mp-22554,MnSO4,6.6864476
mp-22555,RhS2,19.43818
mp-22556,Tm3TlC,23.680729
mp-22557,Ba5Co5O14,15.918461
